# Chatbot Trainer + Evaluation

## Diagram

System

In [1]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
flowchart TD
    %% User Interface Components
    A[User Interface] --> B[Left Sidebar]
    A --> C[Right Chat Area]
    A --> D[Bottom Evaluation Section]

    %% Left Sidebar Components
    B --> E[System Configuration]
    B --> F[Evaluation Configuration]

    E --> G[Preset Dropdown]
    E --> H[System Prompt Input]
    F --> I[Evaluation Metrics Input]

    %% Right Chat Area Components
    C --> J[Chat History Display]
    C --> K[Message Input Box]
    C --> L[Action Buttons]

    L --> M[Send Button]
    L --> N[Reset Chat Button]
    L --> O[🔍 Evaluate Button]

    %% Bottom Section
    D --> P[Collapsible Evaluation Report]

    %% User Actions and Data Flow
    G --> |Select Preset| Q[Load Preset Prompt]
    Q --> H

    K --> |User Types Message| R[User Input]
    M --> |Click Send| S[Chat Function]
    R --> |Enter Key| S

    %% Chat Processing
    S --> T{Has System Prompt?}
    T -->|Yes| U[Add System Message]
    T -->|No| V[Skip System Message]
    U --> W[Add Conversation History]
    V --> W
    W --> X[Add Current User Message]
    X --> Y[OpenAI Chat API Call]

    Y --> Z[Receive Response]
    Z --> AA[Update Chat History]
    AA --> AB[Clear Input Box]
    AB --> J

    %% Evaluation Process
    O --> |Click Evaluate| AC[Start Evaluation]
    AC --> AD[Show Progress: Starting...]
    AD --> AE[Prepare Conversation Transcript]
    AE --> |Progress: 20%| AF[Include System Prompt if exists]
    AF --> AG[Format All Chat Exchanges]
    AG --> |Progress: 40%| AH[Craft Evaluation Prompt]
    AH --> AI[Combine Evaluation Metrics + Conversation]
    AI --> |Progress: 60%| AJ[OpenAI Evaluation API Call]

    AJ --> AK[Receive Evaluation Response]
    AK --> |Progress: 90%| AL[Format Results with Timestamp]
    AL --> |Progress: 100%| AM[Display in Collapsible Section]
    AM --> P

    %% Reset Functionality
    N --> |Click Reset| AN[Clear All Data]
    AN --> AO[Empty Chat History]
    AN --> AP[Clear Message Input]
    AN --> AQ[Reset Evaluation Display]

    %% Error Handling
    Y --> |API Error| AR[Display Error in Chat]
    AJ --> |API Error| AS[Display Error in Evaluation]

    %% Styling for different types of components
    classDef uiComponent fill:#e1f5fe,stroke:#01579b,stroke-width:2px
    classDef apiCall fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef userAction fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef dataFlow fill:#e8f5e8,stroke:#1b5e20,stroke-width:2px
    classDef progressStep fill:#fff8e1,stroke:#f57f17,stroke-width:2px

    class A,B,C,D,J,K,L,P uiComponent
    class Y,AJ apiCall
    class M,N,O,G userAction
    class S,AC,AN dataFlow
    class AD,AE,AF,AH,AI,AK,AL progressStep

'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNqFVl1P4zoQ/StWVzzdLKJAF5aHe+V+0dIWSlN2lzV9MMmktUiTyHaAivLfr2M7qVOQ6AMlmTNnxjNnxn1rBGkIjQvUeEiiOH0JVpRLNO8+JEh9Dg7QnQCOhokEHtEAUCddZ2kCiRQGgUkdsEDfv/+L2mQMkUQ+C+GR8oWFalOHzNhyJVFnRSXCHGjN2iXtVMp0jXrPNM6pZGmCfAiKb4WrknLZP6TU1kw94m+EhLUyJxFb5pxakh2kT5wo+zAD7GngJZlyECBRl6dZmL6UNMY6KCNNucpEqlpkubSIvkYM3UATkJwFooJVh9qry4dzdTTXFdGIARMy5RvUZSKL6WbhQkZkAkLQJZgYqJ2+1uxjgnVFUTtXtU5ElcRYmyfEhyS0xoVruSYzXQadwSf2G/KQh+cnofobHoVlD2EHrc5qu2xba153NceUdNI4pplgjzG4MphBlvJawbTyzFEEoirlLpUU9ZWKDeRSE259iFUYZFq4RbdknNLQPtqe2VPcmn6WIUbGX4eZbzIQyNZ1i2al7Hetnhh0J2bBEyoquEW+6VU/TwJHfDMD7BUjg0awUTjnUNpDZRWoUCxZmve+dpm/DahANbH9924A8wKwvQexRXcEh2GJsgkvXNR1ukW/iP/Ess9RdzrYb02jpuIZuDAdsJqzsF8GZh5+64c/xifnXKnWtKdO/UfD7slNBgkeWq1P1T80jqvO3mvQX6W1ANgzqMYLNQei5PirzVgtniwsxOWOQ7lLzDLBbdKJgfIPY4DN/F85ZXeEZotvTDduV0tBbxHuEF8Wm3LnV3KbGcNd4q/Sl4JsqZQmLpDGq44eHh6WUKN43CvWS0Y51Ms95zQRAWeVPLHZN9sd5/HRgcqlT4ZJEOch1LWBWITgVRVGlP5mG+FL0k/5uih9HJvy9V7V3k+WUCEv9yOd6kgD0uE0knvV2g0QHpg

In [2]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
sequenceDiagram
    participant U as 👤 User
    participant UI as 🖥️ Gradio UI
    participant CS as 💬 Chat System
    participant OAI1 as 🤖 OpenAI Chat API
    participant ES as 📊 Evaluation System
    participant OAI2 as 🔍 OpenAI Evaluation API

    %% Initial Setup
    Note over U, OAI2: 🚀 Application Initialization
    U->>UI: Launch chatbot application
    UI->>UI: Load default system prompt
    UI->>UI: Load default evaluation metrics
    UI-->>U: Display interface ready

    %% Configuration Phase
    Note over U, UI: ⚙️ Configuration Phase
    U->>UI: Select preset from dropdown
    UI->>UI: Load preset system prompt
    UI-->>U: Update system prompt field
    U->>UI: Optionally modify system prompt
    U->>UI: Optionally modify evaluation metrics

    %% Chat Conversation Flow
    Note over U, OAI1: 💬 Chat Conversation Flow
    U->>UI: Type message in input box
    U->>UI: Click Send button (or press Enter)
    UI->>CS: Process chat request

    CS->>CS: Check if system prompt exists
    alt System prompt exists
        CS->>CS: Add system message to context
    end

    CS->>CS: Add conversation history
    CS->>CS: Add current user message
    CS->>OAI1: Send complete message context

    Note over OAI1: Processing request...
    OAI1-->>CS: Return AI response

    CS->>UI: Update chat history
    CS->>UI: Clear input box
    UI-->>U: Display updated conversation

    %% Error handling for chat
    alt API Error during chat
        OAI1-->>CS: Return error
        CS->>UI: Display error message in chat
        UI-->>U: Show error to user
    end

    %% Evaluation Flow
    Note over U, OAI2: 📊 Evaluation Flow
    U->>UI: Click 🔍 Evaluate button
    UI->>ES: Start evaluation process

    ES->>UI: Show "🔄 Evaluating..." message
    UI-->>U: Display progress indicator (0%)

    ES->>ES: Progress: Starting evaluation
    UI-->>U: Update progress (20% - Preparing transcript)

    ES->>ES: Prepare conversation transcript
    ES->>ES: Include system prompt if exists
    UI-->>U: Update progress (40% - Crafting prompt)

    ES->>ES: Format all chat exchanges
    ES->>ES: Craft evaluation prompt
    UI-->>U: Update progress (60% - Sending to OpenAI)

    ES->>ES: Combine evaluation metrics + conversation
    ES->>OAI2: Send evaluation request

    Note over OAI2: Analyzing conversation...
    OAI2-->>ES: Return evaluation response

    UI-->>U: Update progress (90% - Processing results)
    ES->>ES: Format results with timestamp
    ES->>ES: Add conversation statistics

    UI-->>U: Update progress (100% - Complete!)
    ES->>UI: Display formatted evaluation report
    UI-->>U: Show results in collapsible section

    %% Error handling for evaluation
    alt API Error during evaluation
        OAI2-->>ES: Return error
        ES->>UI: Display error message
        UI-->>U: Show evaluation error
    end

    %% Reset Flow
    Note over U, UI: 🔄 Reset Flow
    U->>UI: Click Reset Chat button
    UI->>CS: Clear conversation history
    UI->>UI: Clear message input box
    UI->>UI: Reset evaluation display
    UI-->>U: Display clean interface

    %% Multiple Interactions
    Note over U, OAI2: 🔁 Continuous Usage
    loop Multiple conversations
        U->>UI: Continue chatting...
        Note over UI, OAI1: Repeat chat flow
    end

    U->>UI: Evaluate anytime during conversation
    Note over UI, OAI2: Repeat evaluation flow

'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNqFV9tuGzcQ/RVWgAEbjQ1JdY1YDwEERSkEtLVhVW96oZazEtEVyZLc2EqQf+/wsrvcm2MYNmyeuc+cGX2fZJLBZEEme2HgvxJEBp85PWp63guCX4pqyzOuqLBkR6gh+5J9/I3hz+zhnuwM6AHcpgVk9Pd9mcM0J39oyrjE977MattWTjOyOlFLthdjYcCXp+VmVkuAszJ7IE8KxHITBJfPA1bWHSsZJeuvtCip5VK8Z2vejmjKKluJuLcYpK+uyEZwy2lBtmBLFf77t7RA5FfQZPfBK100KuHjlCyVKngWlEVx/s3/GeR3t58+7TYL8ictRXYiGYZ5kJbQRiziNhVQUkYY5LQsLDE+OqK0PCv7HhCakM5gNc9MjXbwBfnMjSrohXBhQec0A6KBsksS/EqKnB9LHbQ8n6iBgRQ4y/ty/vD4GPtjVKyKfAsFZBZjAAOW5BgKYVoqJl8HQ4+44chDLDvFKLrUgpCcQ8Halp+U84kWxYWcJeP5ZVDrKHgop022XL9i7JgWEzBfCvk63DOzxcCQjMhW7vxzUYBmjaFHwKLhtyotOci3NmyFXfQvplgwciitRVXXUvscGrJ2pb5JcrzaLsizlpl7dI2ILYDsYWwV1mobQasToFaed1IMb9zY2Fi0qOZ86LWlbclYpagKyEqSSXTvLRYB/e854cSyNEkn1C/1ZQhWag049CVSW2UjgYUS+CRl6GoBtklt7Ua3dEEopouLY5Wsu7u7AHWA2+jECxKGRjbZIMooKdwMJA64SsWm9XnvRxJqCVT3Ct0d4NLraacm6cu11tgAJypY4ZzO8Q9nsikaMl4EsVI7SPM8EhU4dKeuzt/KI/+e9mpbYx3B9iRfIxjrX9ZrKCm+C6CZuvGJSlm4sxD6oxRmpLUIIh7i0CQzssa4txYXSTr+KnRB5eW6yoCPaD9JVN/Xroij65RJux1

## Install

In [3]:
pip install gradio openai

In [4]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

## Apps

without history, just one single API call

In [5]:
import gradio as gr
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key = OPENAI_API_KEY)  # Assumes OPENAI_API_KEY is set in environment

def generate_text(inp):
    try:
        # Call OpenAI API for text generation
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # or "gpt-4o" for better quality
            messages=[
                {"role": "user", "content": inp}
            ],
            max_tokens=150,  # Adjust as needed
            temperature=0.7,  # Controls randomness
            top_p=0.9
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Create Gradio interface
gr.Interface(
    fn=generate_text,
    inputs=gr.Textbox(label="Enter your prompt", placeholder="Type your question or prompt here..."),
    outputs=gr.Textbox(label="Generated Response"),
    title="OpenAI Text Generation",
    description="Simple text generation using OpenAI API"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9bcc6adaf035ef510.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


with history

In [6]:
import gradio as gr
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key = OPENAI_API_KEY)  # Assumes OPENAI_API_KEY is set in environment

def chat_with_gpt(message, history):
    # Convert Gradio history format to OpenAI messages format
    messages = []

    # Add conversation history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Add current user message
    messages.append({"role": "user", "content": message})

    try:
        # Call OpenAI API with full conversation context
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # or "gpt-4o" for better quality
            messages=messages,
            max_tokens=500,
            temperature=0.7,
            top_p=0.9
        )

        assistant_response = response.choices[0].message.content

        # Update history with new exchange
        history.append((message, assistant_response))

        return history, ""  # Return updated history and clear input

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        history.append((message, error_msg))
        return history, ""

def reset_conversation():
    return [], ""  # Clear history and input

# Create the Gradio interface
with gr.Blocks(title="OpenAI Chatbot") as demo:
    gr.Markdown("# OpenAI Chatbot")
    gr.Markdown("Chat with GPT-4! Your conversation history is maintained until you reset.")

    # Chatbot component to display conversation
    chatbot = gr.Chatbot(
        label="Conversation",
        value=[],
        height=400
    )

    # Input textbox
    msg_input = gr.Textbox(
        label="Your message",
        placeholder="Type your message here...",
        lines=2,
        scale=4
    )

    # Buttons row
    with gr.Row():
        send_btn = gr.Button("Send", variant="primary", scale=1)
        reset_btn = gr.Button("Reset", variant="secondary", scale=1)

    # Event handlers
    send_btn.click(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input]
    )

    # Allow Enter key to send message
    msg_input.submit(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input]
    )

    # Reset button functionality
    reset_btn.click(
        fn=reset_conversation,
        inputs=[],
        outputs=[chatbot, msg_input]
    )

# Launch the app
demo.launch(share=True)

/tmp/ipython-input-3221534043.py:50: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2757613b5fb692c15.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


with system prompt in sidebar

In [ ]:
import gradio as gr
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key = OPENAI_API_KEY)  # Assumes OPENAI_API_KEY is set in environment

def chat_with_gpt(message, history, system_prompt):
    # Convert Gradio history format to OpenAI messages format
    messages = []

    # Add system prompt if provided
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt.strip()})

    # Add conversation history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Add current user message
    messages.append({"role": "user", "content": message})

    try:
        # Call OpenAI API with full conversation context
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # or "gpt-4o" for better quality
            messages=messages,
            max_tokens=500,
            temperature=0.7,
            top_p=0.9
        )

        assistant_response = response.choices[0].message.content

        # Update history with new exchange
        history.append((message, assistant_response))

        return history, ""  # Return updated history and clear input

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        history.append((message, error_msg))
        return history, ""

def reset_conversation():
    return [], ""  # Clear history and input

def load_preset_prompt(preset):
    """Load predefined system prompts"""
    presets = {
        "General Assistant": "You are a helpful, knowledgeable, and friendly AI assistant.",
        "Therapist": "You are a compassionate and professional therapist. Provide supportive, empathetic responses while maintaining appropriate boundaries. Ask thoughtful questions to help the user explore their feelings.",
        "Distressed Teen": "You are a distressed teenager dealing with typical teenage problems like school stress, peer pressure, and family issues. Respond with the emotional intensity and perspective of a troubled teen seeking help.",
        "Technical Expert": "You are a technical expert with deep knowledge in programming, engineering, and technology. Provide detailed, accurate technical explanations and solutions.",
        "Creative Writer": "You are a creative writing assistant. Help with storytelling, character development, plot ideas, and provide creative inspiration with vivid descriptions.",
        "Custom": ""
    }
    return presets.get(preset, "")

# Create the Gradio interface
with gr.Blocks(title="OpenAI Chatbot with System Prompts") as demo:
    gr.Markdown("# OpenAI Chatbot with System Prompts")

    with gr.Row():
        # Left sidebar for system prompt configuration
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("## System Configuration")

            # Preset dropdown
            preset_dropdown = gr.Dropdown(
                choices=["General Assistant", "Therapist", "Distressed Teen", "Technical Expert", "Creative Writer", "Custom"],
                value="General Assistant",
                label="Quick Presets",
                info="Select a preset or choose 'Custom' to write your own"
            )

            # System prompt textbox
            system_prompt = gr.Textbox(
                label="System Prompt",
                placeholder="Enter system instructions here...",
                value="You are a helpful, knowledgeable, and friendly AI assistant.",
                lines=6,
                info="This guides the AI's behavior and personality"
            )

            gr.Markdown("### Settings")
            gr.Markdown("**Examples:**")
            gr.Markdown("• *General Assistant*: Default helpful AI")
            gr.Markdown("• *Therapist*: For supportive conversations")
            gr.Markdown("• *Distressed Teen*: Practice counseling skills")
            gr.Markdown("• *Technical Expert*: For coding/tech help")
            gr.Markdown("• *Creative Writer*: For creative projects")

        # Right side for chat interface
        with gr.Column(scale=2):
            gr.Markdown("**Chat with your configured AI assistant**")

            # Chatbot component to display conversation
            chatbot = gr.Chatbot(
                label="Conversation",
                value=[],
                height=500
            )

            # Input textbox
            msg_input = gr.Textbox(
                label="Your message",
                placeholder="Type your message here...",
                lines=2,
                scale=4
            )

            # Buttons row
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary", scale=1)
                reset_btn = gr.Button("Reset Chat", variant="secondary", scale=1)

    # Event handlers

    # Load preset prompts
    preset_dropdown.change(
        fn=load_preset_prompt,
        inputs=[preset_dropdown],
        outputs=[system_prompt]
    )

    # Send message
    send_btn.click(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot, system_prompt],
        outputs=[chatbot, msg_input]
    )

    # Allow Enter key to send message
    msg_input.submit(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot, system_prompt],
        outputs=[chatbot, msg_input]
    )

    # Reset button functionality
    reset_btn.click(
        fn=reset_conversation,
        inputs=[],
        outputs=[chatbot, msg_input]
    )

# Launch the app
demo.launch(share=True)

/tmp/ipython-input-7-980155964.py:99: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://66e513f0a9ba264f4f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


with evaluation

In [7]:
import gradio as gr
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key = OPENAI_API_KEY)  # Assumes OPENAI_API_KEY is set in environment

def chat_with_gpt(message, history, system_prompt):
    # Convert Gradio history format to OpenAI messages format
    messages = []

    # Add system prompt if provided
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt.strip()})

    # Add conversation history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Add current user message
    messages.append({"role": "user", "content": message})

    try:
        # Call OpenAI API with full conversation context
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # or "gpt-4o" for better quality
            messages=messages,
            max_tokens=500,
            temperature=0.7,
            top_p=0.9
        )

        assistant_response = response.choices[0].message.content

        # Update history with new exchange
        history.append((message, assistant_response))

        return history, ""  # Return updated history and clear input

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        history.append((message, error_msg))
        return history, ""

def evaluate_conversation(history, system_prompt, evaluation_metrics):
    if not history:
        return "No conversation to evaluate. Please have a conversation first."

    # Prepare the conversation transcript
    conversation_text = ""
    if system_prompt.strip():
        conversation_text += f"System Prompt: {system_prompt}\n\n"

    conversation_text += "Conversation:\n"
    for i, (user_msg, assistant_msg) in enumerate(history, 1):
        conversation_text += f"Turn {i}:\n"
        conversation_text += f"User: {user_msg}\n"
        conversation_text += f"Assistant: {assistant_msg}\n\n"

    # Create evaluation prompt
    evaluation_prompt = f"""Please evaluate the following conversation based on these specific criteria:

{evaluation_metrics}

CONVERSATION TO EVALUATE:
{conversation_text}

Please provide a detailed evaluation report that:
1. Scores each criterion on a scale of 1-10
2. Provides specific examples from the conversation to support your scores
3. Offers constructive feedback for improvement
4. Gives an overall assessment

Format your response clearly with headings for each evaluation criterion."""

    try:
        # Call OpenAI API for evaluation
        response = client.chat.completions.create(
            model="gpt-4o",  # Use better model for evaluation
            messages=[
                {"role": "system", "content": "You are an expert conversation analyst. Provide thorough, objective evaluations with specific examples and actionable feedback."},
                {"role": "user", "content": evaluation_prompt}
            ],
            max_tokens=1000,
            temperature=0.3  # Lower temperature for more consistent evaluation
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error during evaluation: {str(e)}"

def reset_conversation():
    return [], "", ""  # Clear history, input, and evaluation

def load_preset_prompt(preset):
    """Load predefined system prompts"""
    presets = {
        "General Assistant": "You are a helpful, knowledgeable, and friendly AI assistant.",
        "Therapist": "You are a compassionate and professional therapist. Provide supportive, empathetic responses while maintaining appropriate boundaries. Ask thoughtful questions to help the user explore their feelings.",
        "Distressed Teen": "You are a distressed teenager dealing with typical teenage problems like school stress, peer pressure, and family issues. Respond with the emotional intensity and perspective of a troubled teen seeking help.",
        "Technical Expert": "You are a technical expert with deep knowledge in programming, engineering, and technology. Provide detailed, accurate technical explanations and solutions.",
        "Creative Writer": "You are a creative writing assistant. Help with storytelling, character development, plot ideas, and provide creative inspiration with vivid descriptions.",
        "Custom": ""
    }
    return presets.get(preset, "")

# Default evaluation metrics
default_evaluation = """Please evaluate the conversation according to:

1) **Coherence**: How logically consistent and well-structured are the responses? Do they flow naturally from one turn to the next?

2) **Relevance**: How well do the assistant's responses address the user's specific questions, needs, and context?

3) **Engagement**: How natural, conversational, and engaging is the interaction? Does it feel like a meaningful dialogue?

4) **Helpfulness**: How useful and actionable are the assistant's responses? Do they provide value to the user?

5) **Role Consistency**: How well does the assistant maintain its assigned role/persona throughout the conversation? Are there any character breaks?"""

# Create the Gradio interface
with gr.Blocks(title="OpenAI Chatbot with Evaluation") as demo:
    gr.Markdown("# OpenAI Chatbot with Conversation Evaluation")

    with gr.Row():
        # Left sidebar for system prompt and evaluation configuration
        with gr.Column(scale=1, min_width=350):
            gr.Markdown("## System Configuration")

            # Preset dropdown
            preset_dropdown = gr.Dropdown(
                choices=["General Assistant", "Therapist", "Distressed Teen", "Technical Expert", "Creative Writer", "Custom"],
                value="General Assistant",
                label="Quick Presets",
                info="Select a preset or choose 'Custom' to write your own"
            )

            # System prompt textbox
            system_prompt = gr.Textbox(
                label="System Prompt",
                placeholder="Enter system instructions here...",
                value="You are a helpful, knowledgeable, and friendly AI assistant.",
                lines=4,
                info="This guides the AI's behavior and personality"
            )

            gr.Markdown("## Evaluation Configuration")

            # Evaluation metrics textbox
            evaluation_metrics = gr.Textbox(
                label="Evaluation Metrics",
                placeholder="Enter evaluation criteria here...",
                value=default_evaluation,
                lines=8,
                info="Customize how you want the conversation to be evaluated"
            )

            gr.Markdown("### Usage")
            gr.Markdown("• Configure system prompt and evaluation criteria")
            gr.Markdown("• Have a conversation with the AI")
            gr.Markdown("• Click 'Evaluate' to get detailed feedback")

        # Right side for chat interface
        with gr.Column(scale=2):
            gr.Markdown("**Chat with your configured AI assistant**")

            # Chatbot component to display conversation
            chatbot = gr.Chatbot(
                label="Conversation",
                value=[],
                height=500
            )

            # Input textbox
            msg_input = gr.Textbox(
                label="Your message",
                placeholder="Type your message here...",
                lines=2,
                scale=4
            )

            # Buttons row
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary", scale=1)
                reset_btn = gr.Button("Reset Chat", variant="secondary", scale=1)
                evaluate_btn = gr.Button("Evaluate", variant="huggingface", scale=1)

    # Evaluation results section (collapsible)
    with gr.Accordion("📊 Evaluation Report", open=False) as evaluation_accordion:
        evaluation_output = gr.Markdown(
            value="No evaluation yet. Have a conversation and click 'Evaluate' to see detailed feedback.",
            label="Evaluation Results"
        )

    # Event handlers

    # Load preset prompts
    preset_dropdown.change(
        fn=load_preset_prompt,
        inputs=[preset_dropdown],
        outputs=[system_prompt]
    )

    # Send message
    send_btn.click(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot, system_prompt],
        outputs=[chatbot, msg_input]
    )

    # Allow Enter key to send message
    msg_input.submit(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot, system_prompt],
        outputs=[chatbot, msg_input]
    )

    # Evaluate conversation
    evaluate_btn.click(
        fn=evaluate_conversation,
        inputs=[chatbot, system_prompt, evaluation_metrics],
        outputs=[evaluation_output]
    )

    # Reset button functionality
    reset_btn.click(
        fn=reset_conversation,
        inputs=[],
        outputs=[chatbot, msg_input, evaluation_output]
    )

# Launch the app
demo.launch(share=True)

/tmp/ipython-input-3194269163.py:168: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe6d4b3bed3ed1f82b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


with progress bar

In [8]:
import gradio as gr
from openai import OpenAI
import time

# Initialize OpenAI client
client = OpenAI(api_key = OPENAI_API_KEY)  # Assumes OPENAI_API_KEY is set in environment

def chat_with_gpt(message, history, system_prompt):
    # Convert Gradio history format to OpenAI messages format
    messages = []

    # Add system prompt if provided
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt.strip()})

    # Add conversation history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Add current user message
    messages.append({"role": "user", "content": message})

    try:
        # Call OpenAI API with full conversation context
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # or "gpt-4o" for better quality
            messages=messages,
            max_tokens=500,
            temperature=0.7,
            top_p=0.9
        )

        assistant_response = response.choices[0].message.content

        # Update history with new exchange
        history.append((message, assistant_response))

        return history, ""  # Return updated history and clear input

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        history.append((message, error_msg))
        return history, ""

def evaluate_conversation(history, system_prompt, evaluation_metrics, progress=gr.Progress()):
    if not history:
        return "❌ No conversation to evaluate. Please have a conversation first."

    # Initialize progress
    progress(0, desc="Starting evaluation...")

    # Prepare the conversation transcript
    progress(0.2, desc="Preparing conversation transcript...")
    conversation_text = ""
    if system_prompt.strip():
        conversation_text += f"System Prompt: {system_prompt}\n\n"

    conversation_text += "Conversation:\n"
    for i, (user_msg, assistant_msg) in enumerate(history, 1):
        conversation_text += f"Turn {i}:\n"
        conversation_text += f"User: {user_msg}\n"
        conversation_text += f"Assistant: {assistant_msg}\n\n"

    # Create evaluation prompt
    progress(0.4, desc="Crafting evaluation prompt...")
    evaluation_prompt = f"""Please evaluate the following conversation based on these specific criteria:

{evaluation_metrics}

CONVERSATION TO EVALUATE:
{conversation_text}

Please provide a detailed evaluation report that:
1. Scores each criterion on a scale of 1-10
2. Provides specific examples from the conversation to support your scores
3. Offers constructive feedback for improvement
4. Gives an overall assessment

Format your response clearly with headings for each evaluation criterion."""

    try:
        # Call OpenAI API for evaluation
        progress(0.6, desc="Sending request to OpenAI...")
        response = client.chat.completions.create(
            model="gpt-4o",  # Use better model for evaluation
            messages=[
                {"role": "system", "content": "You are an expert conversation analyst. Provide thorough, objective evaluations with specific examples and actionable feedback."},
                {"role": "user", "content": evaluation_prompt}
            ],
            max_tokens=1000,
            temperature=0.3  # Lower temperature for more consistent evaluation
        )

        progress(0.9, desc="Processing evaluation results...")

        # Format the response nicely
        evaluation_result = f"""# 📊 Conversation Evaluation Report

{response.choices[0].message.content}

---
*Evaluation completed at {time.strftime('%Y-%m-%d %H:%M:%S')}*
*Conversation length: {len(history)} exchanges*
"""

        progress(1.0, desc="Evaluation complete!")
        return evaluation_result

    except Exception as e:
        progress(1.0, desc="Evaluation failed")
        return f"❌ **Error during evaluation:** {str(e)}"

def start_evaluation():
    """Return initial evaluation status"""
    return "🔄 **Evaluating conversation...** \n\nPlease wait while we analyze your conversation. This may take 10-30 seconds depending on conversation length."

def reset_conversation():
    return [], "", "No evaluation yet. Have a conversation and click 'Evaluate' to see detailed feedback."  # Clear history, input, and evaluation

def load_preset_prompt(preset):
    """Load predefined system prompts"""
    presets = {
        "General Assistant": "You are a helpful, knowledgeable, and friendly AI assistant.",
        "Therapist": "You are a compassionate and professional therapist. Provide supportive, empathetic responses while maintaining appropriate boundaries. Ask thoughtful questions to help the user explore their feelings.",
        "Distressed Teen": "You are a distressed teenager dealing with typical teenage problems like school stress, peer pressure, and family issues. Respond with the emotional intensity and perspective of a troubled teen seeking help.",
        "Technical Expert": "You are a technical expert with deep knowledge in programming, engineering, and technology. Provide detailed, accurate technical explanations and solutions.",
        "Creative Writer": "You are a creative writing assistant. Help with storytelling, character development, plot ideas, and provide creative inspiration with vivid descriptions.",
        "Custom": ""
    }
    return presets.get(preset, "")

# Default evaluation metrics
default_evaluation = """Please evaluate the conversation according to:

1) **Coherence**: How logically consistent and well-structured are the responses? Do they flow naturally from one turn to the next?

2) **Relevance**: How well do the assistant's responses address the user's specific questions, needs, and context?

3) **Engagement**: How natural, conversational, and engaging is the interaction? Does it feel like a meaningful dialogue?

4) **Helpfulness**: How useful and actionable are the assistant's responses? Do they provide value to the user?

5) **Role Consistency**: How well does the assistant maintain its assigned role/persona throughout the conversation? Are there any character breaks?"""

# Create the Gradio interface
with gr.Blocks(title="OpenAI Chatbot with Evaluation") as demo:
    gr.Markdown("# OpenAI Chatbot with Conversation Evaluation")

    with gr.Row():
        # Left sidebar for system prompt and evaluation configuration
        with gr.Column(scale=1, min_width=350):
            gr.Markdown("## System Configuration")

            # Preset dropdown
            preset_dropdown = gr.Dropdown(
                choices=["General Assistant", "Therapist", "Distressed Teen", "Technical Expert", "Creative Writer", "Custom"],
                value="General Assistant",
                label="Quick Presets",
                info="Select a preset or choose 'Custom' to write your own"
            )

            # System prompt textbox
            system_prompt = gr.Textbox(
                label="System Prompt",
                placeholder="Enter system instructions here...",
                value="You are a helpful, knowledgeable, and friendly AI assistant.",
                lines=4,
                info="This guides the AI's behavior and personality"
            )

            gr.Markdown("## Evaluation Configuration")

            # Evaluation metrics textbox
            evaluation_metrics = gr.Textbox(
                label="Evaluation Metrics",
                placeholder="Enter evaluation criteria here...",
                value=default_evaluation,
                lines=8,
                info="Customize how you want the conversation to be evaluated"
            )

            gr.Markdown("### Usage")
            gr.Markdown("• Configure system prompt and evaluation criteria")
            gr.Markdown("• Have a conversation with the AI")
            gr.Markdown("• Click 'Evaluate' to get detailed feedback")
            gr.Markdown("• Evaluation takes 10-30 seconds ⏱️")

        # Right side for chat interface
        with gr.Column(scale=2):
            gr.Markdown("**Chat with your configured AI assistant**")

            # Chatbot component to display conversation
            chatbot = gr.Chatbot(
                label="Conversation",
                value=[],
                height=500
            )

            # Input textbox
            msg_input = gr.Textbox(
                label="Your message",
                placeholder="Type your message here...",
                lines=2,
                scale=4
            )

            # Buttons row
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary", scale=1)
                reset_btn = gr.Button("Reset Chat", variant="secondary", scale=1)
                evaluate_btn = gr.Button("🔍 Evaluate", variant="huggingface", scale=1)

    # Evaluation results section (collapsible)
    with gr.Accordion("📊 Evaluation Report", open=False) as evaluation_accordion:
        evaluation_output = gr.Markdown(
            value="No evaluation yet. Have a conversation and click 'Evaluate' to see detailed feedback.",
            label="Evaluation Results"
        )

    # Event handlers

    # Load preset prompts
    preset_dropdown.change(
        fn=load_preset_prompt,
        inputs=[preset_dropdown],
        outputs=[system_prompt]
    )

    # Send message
    send_btn.click(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot, system_prompt],
        outputs=[chatbot, msg_input]
    )

    # Allow Enter key to send message
    msg_input.submit(
        fn=chat_with_gpt,
        inputs=[msg_input, chatbot, system_prompt],
        outputs=[chatbot, msg_input]
    )

    # Evaluate conversation with progress tracking
    evaluate_btn.click(
        fn=start_evaluation,
        inputs=[],
        outputs=[evaluation_output]
    ).then(
        fn=evaluate_conversation,
        inputs=[chatbot, system_prompt, evaluation_metrics],
        outputs=[evaluation_output]
    )

    # Reset button functionality
    reset_btn.click(
        fn=reset_conversation,
        inputs=[],
        outputs=[chatbot, msg_input, evaluation_output]
    )

# Launch the app
demo.launch(share=True)

/tmp/ipython-input-1688659477.py:194: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de5e97fd0c33a64a27.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
